In [9]:
from pyspark.sql import functions as F

# Step 1: Read existing Gold DimZone
old_zone = spark.read.table("DimZone")

# Step 2: Read official cleaned Zone Lookup from Silver
lookup_zone = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.taxi_zone_lookup_clean"
)

print("Existing Gold zones:", old_zone.count())
print("Official lookup records:", lookup_zone.count())

# Step 3: Join existing Gold locations with official lookup
updated_zone = (
    old_zone.alias("d")
    .join(
        lookup_zone.alias("l"),
        F.col("d.location_id") == F.col("l.location_id"),
        "left"
    )
    .select(
        F.col("d.zone_key").cast("int").alias("zone_key"),
        F.col("d.location_id").cast("int").alias("location_id"),
        F.coalesce(
            F.col("l.borough"),
            F.lit("Unknown")
        ).alias("borough"),
        F.coalesce(
            F.col("l.zone_name"),
            F.concat(
                F.lit("Location "),
                F.col("d.location_id").cast("string")
            )
        ).alias("zone_name"),
        F.coalesce(
            F.col("l.service_zone"),
            F.lit("Unknown")
        ).alias("service_zone")
    )
)

# Step 4: Remove duplicate locations if any
updated_zone = updated_zone.dropDuplicates(["location_id"])

print("Prepared Gold zones:", updated_zone.count())

# Step 5: Save updated DimZone to Gold Lakehouse
(
    updated_zone.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("DimZone")
)

print("Gold DimZone updated successfully!")
print("Total Gold zones:", updated_zone.count())

# Step 6: Verify updated table
final_zone = spark.read.table("DimZone")

print("Verified Gold zones:", final_zone.count())

print("Remaining Unknown Boroughs:")
final_zone.filter(
    F.col("borough") == "Unknown"
).show(truncate=False)

display(final_zone.orderBy("zone_key"))

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 11, Finished, Available, Finished, False)

Existing Gold zones: 262
Official lookup records: 263
Prepared Gold zones: 262
Gold DimZone updated successfully!
Total Gold zones: 262
Verified Gold zones: 262
Remaining Unknown Boroughs:
+--------+-----------+-------+------------+------------+
|zone_key|location_id|borough|zone_name   |service_zone|
+--------+-----------+-------+------------+------------+
|264     |264        |Unknown|Location 264|Unknown     |
|265     |265        |Unknown|Location 265|Unknown     |
+--------+-----------+-------+------------+------------+



SynapseWidget(Synapse.DataFrame, 4c4998c5-62fc-4327-ab5a-9356b19ebeb1)

In [10]:
from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiDaily")

print("FactTaxiDaily columns:")
print(fact.columns)

print("DimZone unknown locations:")
display(
    spark.read.table("DimZone")
    .filter(F.col("location_id").isin(264, 265))
)

print("FactTaxiDaily schema:")
fact.printSchema()

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 12, Finished, Available, Finished, False)

FactTaxiDaily columns:
['date', 'pickup_location_id', 'total_trips', 'total_passengers', 'total_distance', 'total_revenue', 'avg_trip_distance', 'avg_trip_duration_minutes', 'avg_fare', 'year', 'month', 'day', 'pickup_hour']
DimZone unknown locations:


SynapseWidget(Synapse.DataFrame, 184be0e3-2b27-46da-9f3c-abe0f4fe3fd6)

FactTaxiDaily schema:
root
 |-- date: date (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_passengers: long (nullable = true)
 |-- total_distance: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_trip_duration_minutes: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)



In [11]:
from pyspark.sql import functions as F

# Read Gold tables
fact = spark.read.table("FactTaxiDaily")
dim = spark.read.table("DimZone")

# Check whether these locations exist in FactTaxiDaily
location_check = (
    fact
    .filter(F.col("pickup_location_id").isin(264, 265))
    .groupBy("pickup_location_id")
    .agg(
        F.sum("total_trips").alias("total_trips"),
        F.sum("total_passengers").alias("total_passengers"),
        F.sum("total_revenue").alias("total_revenue")
    )
    .orderBy("pickup_location_id")
)

print("Trip records for locations 264 and 265:")
display(location_check)

# Check all location IDs in DimZone
print("Unknown locations in DimZone:")
display(
    dim
    .filter(F.col("borough") == "Unknown")
    .select(
        "zone_key",
        "location_id",
        "borough",
        "zone_name",
        "service_zone"
    )
    .orderBy("location_id")
)

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 13, Finished, Available, Finished, False)

Trip records for locations 264 and 265:


SynapseWidget(Synapse.DataFrame, d58c5c0a-d262-4546-bf21-1a9b6ed6360f)

Unknown locations in DimZone:


SynapseWidget(Synapse.DataFrame, a4454723-9b75-440d-b674-80806f0c1a85)

In [12]:
fact = spark.read.table("FactTaxiDaily")

fact.filter(
    fact.pickup_location_id.isin(264, 265)
).count()

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 14, Finished, Available, Finished, False)

62

In [13]:
from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiDaily")

display(
    fact
    .filter(F.col("pickup_location_id").isin(264, 265))
    .groupBy("pickup_location_id")
    .agg(
        F.sum("total_trips").alias("actual_total_trips"),
        F.sum("total_passengers").alias("total_passengers"),
        F.sum("total_revenue").alias("total_revenue")
    )
    .orderBy("pickup_location_id")
)

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b6934965-c428-4924-bff2-7f23121ce611)

In [14]:
from pyspark.sql import functions as F

dim = spark.read.table("DimZone")
fact = spark.read.table("FactTaxiDaily")

# Check duplicate location IDs
print("Duplicate location IDs:")
display(
    dim.groupBy("location_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check fact locations missing from DimZone
fact_locations = fact.select(
    F.col("pickup_location_id").alias("location_id")
).distinct()

dim_locations = dim.select("location_id").distinct()

missing_locations = fact_locations.join(
    dim_locations,
    on="location_id",
    how="left_anti"
)

print("Fact locations missing from DimZone:")
display(missing_locations)

# Final counts
print("DimZone rows:", dim.count())
print("Distinct location IDs:", dim.select("location_id").distinct().count())
print("Unknown zones:", dim.filter(F.col("borough") == "Unknown").count())

StatementMeta(, 148578f9-58ba-4c36-a8b7-5f18ded640bb, 16, Finished, Available, Finished, False)

Duplicate location IDs:


SynapseWidget(Synapse.DataFrame, df4fa22e-52ce-4529-8cc2-4838e196f554)

Fact locations missing from DimZone:


SynapseWidget(Synapse.DataFrame, 000d8f93-2cc1-407d-a9bf-926ebbbea7d7)

DimZone rows: 262
Distinct location IDs: 262
Unknown zones: 2
